# INFORMATIVE 문장 5개 카테고리 분류

WNUT-2020 Task 2 데이터셋에서 `INFORMATIVE` 문장만 골라서 5개 카테고리로 분류합니다.

- `confirmed`: 확진 정보
- `suspected`: 의심/증상 정보
- `death`: 사망 정보
- `recovered`: 회복/퇴원 정보
- `tested`: 검사 정보


In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import re
from pathlib import Path

#업로드된 파일 확인
list(uploaded.keys())

Saving train.tsv to train.tsv


['train.tsv']

In [ ]:
input_file = list(uploaded.keys())[0]
print(input_file)

if input_file.endswith(".csv"):
    df = pd.read_csv(input_file)
else:
    df = pd.read_csv(input_file, sep="\t")

df.head()

train.tsv


,Id,Text,Label
0,1241490299215634434,Official death toll from #covid19 in the Unite...,INFORMATIVE
1,1245916400981381130,"Dearest Mr. President @USER 1,169 coronavirus ...",INFORMATIVE
2,1241132432402849793,Latest Updates March 20 ⚠️5274 new cases and 3...,INFORMATIVE
3,1236107253666607104,真把公主不当干部 BREAKING: 21 people on Grand Princess...,INFORMATIVE
4,1239673817552879619,OKLAHOMA CITY — The State Department of Educat...,UNINFORMATIVE


In [ ]:
CATEGORIES = {
    "confirmed": [
        r"\bconfirmed\b",
        r"\bconfirm(?:s|ed|ing)?\b",
        r"\bpositive\b",
        r"\btested positive\b",
        r"\bnew cases?\b",
    ],
    "suspected": [
        r"\bsuspected\b",
        r"\bsuspect(?:s|ed|ing)?\b",
        r"\bsymptoms?\b",
        r"\bsymptomatic\b",
        r"\bunder observation\b",
    ],
    "death": [
        r"\bdeaths?\b",
        r"\bdied\b",
        r"\bdies\b",
        r"\bfatalit(?:y|ies)\b",
        r"\bdeceased\b",
    ],
    "recovered": [
        r"\brecovered\b",
        r"\brecover(?:s|ed|ing|ies)?\b",
        r"\bdischarged\b",
        r"\bdischarge(?:s|d)?\b",
    ],
    "tested": [
        r"\btested\b",
        r"\btesting\b",
        r"\btests?\b",
        r"\bscreened\b",
        r"\bscreening\b",
    ],
}

In [ ]:
def classify_info_category(text):
    text = str(text).lower()
    matched = []

    for category, patterns in CATEGORIES.items():
        for pattern in patterns:
            if re.search(pattern, text):
                matched.append(category)
                break

    if matched:
        return ";".join(matched)
    return "other_informative"

In [ ]:
print(df.columns)

Index(['Id', 'Text', 'Label'], dtype='object')


In [ ]:
text_col = "Text"
label_col = "Label"

info_df = df[df[label_col].astype(str).str.upper() == "INFORMATIVE"].copy()
info_df["info_categories"] = info_df[text_col].apply(classify_info_category)

info_df.head()

,Id,Text,Label,info_categories
0,1241490299215634434,Official death toll from #covid19 in the Unite...,INFORMATIVE,death
1,1245916400981381130,"Dearest Mr. President @USER 1,169 coronavirus ...",INFORMATIVE,death
2,1241132432402849793,Latest Updates March 20 ⚠️5274 new cases and 3...,INFORMATIVE,confirmed;death
3,1236107253666607104,真把公主不当干部 BREAKING: 21 people on Grand Princess...,INFORMATIVE,confirmed;tested
6,1249147011003187200,"as number of #COVID19 deaths surpassed 100,000...",INFORMATIVE,death


In [ ]:
category_counts = (
    info_df["info_categories"]
    .str.get_dummies(sep=";")
    .sum()
    .sort_values(ascending=False)
)

category_counts



,0
confirmed,1402
death,1269
tested,883
other_informative,532
recovered,282
suspected,160


In [ ]:
category_ratio = category_counts / len(info_df) * 100
category_ratio

,0
confirmed,42.835319
death,38.771769
tested,26.978307
other_informative,16.254201
recovered,8.615949
suspected,4.888482
